# Generateur de contenu personnalise base sur l'IA

Ce notebook traite le sujet de maniere simple et complete : generation de profils utilisateurs, nettoyage, analyse statistique, visualisation et moteur de recommandation personnalise.

Bibliotheques utilisees uniquement : Pandas, NumPy, Matplotlib, Seaborn et SciPy.

## 1. Importation des outils

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp

sns.set_theme(style="whitegrid")
np.random.seed(42)

## 2. Definition des profils et preferences utilisateurs

On prepare des variables, listes, dictionnaires et categories de contenu. Ces structures serviront a generer un jeu de donnees synthetique.

In [ ]:
n_users = 80

first_names = [
    "Amina", "Lucas", "Nora", "Yanis", "Ines", "Hugo", "Sara", "Adam",
    "Lina", "Mehdi", "Emma", "Noah", "Lea", "Ilyes", "Maya", "Tom"
]

interests_catalog = ["technology", "fitness", "music", "books", "cooking", "travel", "gaming"]

# La distribution rend volontairement fitness et music plus frequents que technology.
interest_probabilities = np.array([0.16, 0.24, 0.22, 0.13, 0.08, 0.10, 0.07])

activities_by_interest = {
    "technology": ["watched AI talk", "read tech blog", "bought keyboard", "bought headphones"],
    "fitness": ["watched workout video", "liked yoga tip", "bought running shoes", "logged morning run"],
    "music": ["listened to rock music", "liked jazz playlist", "bought concert ticket", "shared pop playlist"],
    "books": ["read mystery review", "liked science fiction", "bought novel", "saved reading list"],
    "cooking": ["watched pasta recipe", "liked vegan recipe", "bought cooking pan", "saved dessert idea"],
    "travel": ["viewed beach guide", "liked city itinerary", "bought travel backpack", "saved hotel idea"],
    "gaming": ["watched game stream", "liked strategy guide", "bought game controller", "joined gaming forum"]
}

content_library = {
    "technology": ["Article IA pour debutants", "Blog sur les objets connectes", "Podcast tendances tech"],
    "fitness": ["Programme cardio 15 minutes", "Conseil nutrition du jour", "Routine yoga simple"],
    "music": ["Playlist rock populaire", "Selection jazz calme", "Nouveautes pop de la semaine"],
    "books": ["Roman recommande du mois", "Liste science-fiction", "Essai court a decouvrir"],
    "cooking": ["Recette rapide du soir", "Idee dessert facile", "Menu sain de la semaine"],
    "travel": ["Guide week-end en ville", "Checklist voyage leger", "Destination nature a explorer"],
    "gaming": ["Guide strategie debutant", "Top jeux cooperatifs", "Video gameplay conseillee"]
}

print("Exemple de profil attendu :")
example_user = {
    "name": "John Doe",
    "age": 28,
    "interests": ["technology", "music"],
    "activity_log": ["watched AI talk", "listened to rock music", "bought headphones"]
}
print(example_user)

## 3. Generation du jeu de donnees synthetique

On cree deux DataFrames : un pour les utilisateurs et un pour les activites. On ajoute volontairement quelques doublons et valeurs manquantes pour montrer le pretraitement.

In [ ]:
profiles = []
activity_rows = []

hours = np.arange(24)
hour_probabilities = np.array([
    0.01, 0.01, 0.01, 0.01, 0.01, 0.02,
    0.03, 0.04, 0.04, 0.04, 0.04, 0.04,
    0.05, 0.05, 0.05, 0.05, 0.06, 0.07,
    0.08, 0.09, 0.09, 0.08, 0.05, 0.02
])
hour_probabilities = hour_probabilities / hour_probabilities.sum()

for i in range(n_users):
    user_id = i + 1
    name = first_names[i % len(first_names)] + " " + str(user_id)
    age = int(np.random.randint(16, 61))
    number_of_interests = int(np.random.choice([1, 2, 3], p=[0.25, 0.50, 0.25]))
    interests = list(np.random.choice(interests_catalog, size=number_of_interests, replace=False, p=interest_probabilities))
    number_of_actions = int(np.random.randint(3, 9))
    activity_log = []

    for action_index in range(number_of_actions):
        category = str(np.random.choice(interests))
        activity = str(np.random.choice(activities_by_interest[category]))
        action_type = activity.split()[0]
        hour = int(np.random.choice(hours, p=hour_probabilities))
        activity_log.append(activity)

        activity_rows.append({
            "user_id": user_id,
            "name": name,
            "category": category,
            "activity": activity,
            "action_type": action_type,
            "hour": hour
        })

    profiles.append({
        "user_id": user_id,
        "name": name,
        "age": age,
        "interests": interests,
        "activity_log": activity_log
    })

df_users = pd.DataFrame(profiles)
df_activities = pd.DataFrame(activity_rows)

# Ajout de petits problemes realistes : doublons et valeurs manquantes.
df_users = pd.concat([df_users, df_users.iloc[[2]]], ignore_index=True)
df_activities = pd.concat([df_activities, df_activities.iloc[[0]]], ignore_index=True)
df_users.loc[3, "age"] = np.nan
df_users.loc[10, "interests"] = np.nan
df_activities.loc[4, "hour"] = np.nan

print(df_users.head(11).to_string())
print("\nShape utilisateurs :", df_users.shape)
print("\nTypes utilisateurs :")
print(df_users.dtypes)
print("\nDescription utilisateurs :")
print(df_users.describe())
print("\nValeurs manquantes utilisateurs :")
print(df_users.isnull().sum())

print("\nShape activites :", df_activities.shape)
print("\nValeurs manquantes activites :")
print(df_activities.isnull().sum())

## 4. Nettoyage, pretraitement et ingenierie des caracteristiques

On supprime les doublons, on remplace les valeurs manquantes, puis on ajoute des variables utiles a l'analyse.

In [ ]:
df_users_clean = df_users.drop_duplicates(subset="user_id").copy()
df_activities_clean = df_activities.drop_duplicates().copy()

median_age = int(df_users_clean["age"].median())
median_hour = int(df_activities_clean["hour"].median())

df_users_clean["age"] = df_users_clean["age"].fillna(median_age).astype(int)
df_activities_clean["hour"] = df_activities_clean["hour"].fillna(median_hour).astype(int)

df_users_clean["interests"] = df_users_clean["interests"].apply(
    lambda value: value if isinstance(value, list) and len(value) > 0 else ["unknown"]
)
df_users_clean["activity_log"] = df_users_clean["activity_log"].apply(
    lambda value: value if isinstance(value, list) else []
)

df_users_clean["interest_count"] = df_users_clean["interests"].apply(len)
df_users_clean["activity_count"] = df_users_clean["activity_log"].apply(len)
df_users_clean["main_interest"] = df_users_clean["interests"].apply(lambda values: values[0])
df_users_clean["age_segment"] = pd.cut(
    df_users_clean["age"],
    bins=[15, 24, 34, 49, 100],
    labels=["16-24", "25-34", "35-49", "50+"]
)

df_activities_clean["period"] = pd.cut(
    df_activities_clean["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["night", "morning", "afternoon", "evening"]
)

print("Utilisateurs nettoyes :", df_users_clean.shape)
print("Activites nettoyees :", df_activities_clean.shape)
print("\nApercu apres nettoyage :")
print(df_users_clean.head().to_string())

## 5. Fonctions, lambda, map, filter et reduce

Cette cellule montre les notions essentielles de Python avec des exemples simples appliques aux donnees.

In [ ]:
def clean_text(text):
    return str(text).strip().lower()

def simple_reduce(function, sequence, initial_value):
    result = initial_value
    for element in sequence:
        result = function(result, element)
    return result

get_first_interest = lambda interests: interests[0] if len(interests) > 0 else "unknown"

main_interests_with_map = list(map(get_first_interest, df_users_clean["interests"]))
very_active_users = list(filter(lambda user: user["activity_count"] >= 6, df_users_clean.to_dict("records")))
total_actions = simple_reduce(lambda total, value: total + value, df_users_clean["activity_count"].tolist(), 0)

interest_counter = {}
for interests in df_users_clean["interests"]:
    for interest in interests:
        interest_counter[interest] = interest_counter.get(interest, 0) + 1

print("Exemple clean_text :", clean_text("  Music  "))
print("5 premiers interets principaux avec map :", main_interests_with_map[:5])
print("Nombre d'utilisateurs tres actifs avec filter :", len(very_active_users))
print("Total des actions avec reduce simplifie :", total_actions)
print("Comptage avec dictionnaire et boucles :", interest_counter)

## 6. Analyse exploratoire des donnees

On identifie les centres d'interet les plus courants, les categories les plus actives et les heures de pic.

In [ ]:
interest_distribution = df_users_clean.explode("interests")["interests"].value_counts()
category_distribution = df_activities_clean["category"].value_counts()
hour_distribution = df_activities_clean["hour"].value_counts().sort_index()
period_distribution = df_activities_clean["period"].value_counts()

peak_hour = int(hour_distribution.idxmax())
most_common_interest = interest_distribution.index[0]
most_active_category = category_distribution.index[0]

print("Interet le plus courant :", most_common_interest)
print("Categorie la plus active :", most_active_category)
print("Heure de pic d'activite :", peak_hour, "h")
print("\nDistribution des interets :")
print(interest_distribution)
print("\nDistribution des periodes :")
print(period_distribution)

## 7. Analyse statistique avec SciPy

On utilise SciPy pour tester des distributions et analyser une relation entre deux comportements : regarder un contenu IA et acheter un produit technologique.

In [ ]:
observed_interests = interest_distribution.reindex(interests_catalog, fill_value=0)
expected_interests = observed_interests.sum() * interest_probabilities
chi_interest = sp.stats.chisquare(f_obs=observed_interests, f_exp=expected_interests)

observed_actions = df_activities_clean["action_type"].value_counts().sort_index()
expected_actions = np.repeat(observed_actions.sum() / len(observed_actions), len(observed_actions))
chi_actions = sp.stats.chisquare(f_obs=observed_actions, f_exp=expected_actions)

user_flags = df_activities_clean.groupby("user_id").agg(
    watched_ai=("activity", lambda values: int("watched AI talk" in list(values))),
    bought_tech=("activity", lambda values: int(any(item in list(values) for item in ["bought keyboard", "bought headphones"])))
)

contingency_table = pd.crosstab(user_flags["watched_ai"], user_flags["bought_tech"])
contingency_table = contingency_table.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
chi2_value, p_value, dof, expected_table = sp.stats.chi2_contingency(contingency_table)

activity_loc, activity_scale = sp.stats.norm.fit(df_users_clean["activity_count"])

print("Test chi-carre sur la distribution des interets")
print("statistique =", round(chi_interest.statistic, 3), "p-value =", round(chi_interest.pvalue, 3))

print("\nTest chi-carre sur les types d'actions")
print("statistique =", round(chi_actions.statistic, 3), "p-value =", round(chi_actions.pvalue, 3))

print("\nTable de contingence : watched_ai x bought_tech")
print(contingency_table)
print("chi2 =", round(chi2_value, 3), "p-value =", round(p_value, 3), "dof =", dof)

if p_value < 0.05:
    print("Conclusion : relation statistiquement significative au seuil de 5%.")
else:
    print("Conclusion : pas de relation statistiquement significative au seuil de 5%.")

print("\nAjustement normal du nombre d'activites par utilisateur :")
print("moyenne estimee =", round(activity_loc, 2), "ecart-type estime =", round(activity_scale, 2))

## 8. Modele sequentiel simple

On simule une logique d'IA generative simple : selon la derniere categorie consultee, le systeme predit la prochaine categorie probable.

In [ ]:
sequence_pairs = []

for user_id, user_activities in df_activities_clean.sort_values(["user_id", "hour"]).groupby("user_id"):
    categories = user_activities["category"].tolist()
    for current_category, next_category in zip(categories[:-1], categories[1:]):
        sequence_pairs.append({
            "current_category": current_category,
            "next_category": next_category
        })

df_sequences = pd.DataFrame(sequence_pairs)
transition_matrix = pd.crosstab(
    df_sequences["current_category"],
    df_sequences["next_category"],
    normalize="index"
)

def predict_next_category(last_category):
    if last_category in transition_matrix.index:
        return transition_matrix.loc[last_category].idxmax()
    return most_common_interest

print("Matrice de transition simple :")
print(transition_matrix.round(2))
print("\nExemple : apres technology, categorie probable =", predict_next_category("technology"))

## 9. Moteur de recommandation avec POO

On utilise l'encapsulation, l'heritage et le polymorphisme. Le moteur combine les interets, l'activite passee, les utilisateurs similaires et le modele sequentiel.

In [ ]:
interest_features = pd.DataFrame(0.0, index=df_users_clean["user_id"], columns=interests_catalog)

for _, user in df_users_clean.iterrows():
    for interest in user["interests"]:
        if interest in interest_features.columns:
            interest_features.loc[user["user_id"], interest] = 1.0

activity_features = pd.crosstab(df_activities_clean["user_id"], df_activities_clean["category"])
activity_features = activity_features.reindex(index=df_users_clean["user_id"], columns=interests_catalog, fill_value=0)
activity_features = activity_features.div(activity_features.max().replace(0, 1))

feature_matrix = interest_features + activity_features

class BaseRecommender:
    def __init__(self, content_library):
        self.__content_library = content_library

    def get_content(self, category):
        return self.__content_library.get(category, [])

    def recommend(self, user, n=3):
        recommendations = []
        for interest in user["interests"]:
            recommendations.extend(self.get_content(interest))
        return list(dict.fromkeys(recommendations))[:n]

class ActivityAwareRecommender(BaseRecommender):
    def recommend(self, user, n=3):
        recommendations = super().recommend(user, n=10)
        activity_text = " ".join(user["activity_log"]).lower()

        if "ai" in activity_text or "tech" in activity_text:
            recommendations = self.get_content("technology") + recommendations
        if "music" in activity_text or "playlist" in activity_text:
            recommendations = self.get_content("music") + recommendations
        if user["activity_count"] >= 6:
            recommendations = recommendations + self.get_content(user["main_interest"])

        return list(dict.fromkeys(recommendations))[:n]

class SimilarUserRecommender(ActivityAwareRecommender):
    def __init__(self, content_library, feature_matrix, users_df):
        super().__init__(content_library)
        self.feature_matrix = feature_matrix
        self.users_df = users_df.set_index("user_id")

    def find_similar_user_id(self, user_id):
        target_vector = self.feature_matrix.loc[user_id].values
        distances = []

        for other_user_id, row in self.feature_matrix.iterrows():
            if other_user_id != user_id:
                distance_value = sp.spatial.distance.cosine(target_vector, row.values)
                if np.isnan(distance_value):
                    distance_value = 1.0
                distances.append((other_user_id, distance_value))

        return sorted(distances, key=lambda item: item[1])[0][0]

    def recommend(self, user, n=3):
        recommendations = super().recommend(user, n=10)
        similar_user_id = self.find_similar_user_id(user["user_id"])
        similar_user = self.users_df.loc[similar_user_id]

        for interest in similar_user["interests"]:
            recommendations.extend(self.get_content(interest))

        user_history = df_activities_clean[df_activities_clean["user_id"] == user["user_id"]].sort_values("hour")
        if len(user_history) > 0:
            last_category = user_history.iloc[-1]["category"]
            next_category = predict_next_category(last_category)
            recommendations.extend(self.get_content(next_category))

        return list(dict.fromkeys(recommendations))[:n]

engine = SimilarUserRecommender(content_library, feature_matrix, df_users_clean)

for user in df_users_clean.head(5).to_dict("records"):
    print("Utilisateur :", user["name"])
    print("Interets :", user["interests"])
    print("Suggestions :", engine.recommend(user, n=3))
    print("-")

## 10. Table finale des recommandations

On genere des recommandations pour tous les utilisateurs afin de pouvoir analyser les categories les plus recommandees par segment.

In [ ]:
recommendation_rows = []

for user in df_users_clean.to_dict("records"):
    suggestions = engine.recommend(user, n=3)
    for suggestion in suggestions:
        suggested_category = "unknown"
        for category, items in content_library.items():
            if suggestion in items:
                suggested_category = category
        recommendation_rows.append({
            "user_id": user["user_id"],
            "name": user["name"],
            "age_segment": user["age_segment"],
            "recommendation": suggestion,
            "recommended_category": suggested_category
        })

df_recommendations = pd.DataFrame(recommendation_rows)
recommendations_by_segment = df_recommendations.groupby(
    ["age_segment", "recommended_category"],
    observed=True
).size().reset_index(name="count")

print(df_recommendations.head(12).to_string())
print("\nCategories les plus recommandees :")
print(df_recommendations["recommended_category"].value_counts())

## 11. Visualisations avec Matplotlib et Seaborn

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(x=interest_distribution.index, y=interest_distribution.values)
plt.title("Repartition des centres d'interet")
plt.xlabel("Centre d'interet")
plt.ylabel("Nombre d'utilisateurs")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

activity_heatmap = pd.crosstab(df_activities_clean["category"], df_activities_clean["hour"])
plt.figure(figsize=(13, 5))
sns.heatmap(activity_heatmap, cmap="YlGnBu", linewidths=0.2)
plt.title("Intensite d'activite par heure et par categorie")
plt.xlabel("Heure")
plt.ylabel("Categorie")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
sns.barplot(
    data=recommendations_by_segment,
    x="age_segment",
    y="count",
    hue="recommended_category"
)
plt.title("Categories les plus recommandees par segment d'age")
plt.xlabel("Segment d'age")
plt.ylabel("Nombre de recommandations")
plt.legend(title="Categorie", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 12. Simulation finale de generation de contenu personnalise

On selectionne un utilisateur et on affiche une reponse personnalisee comme le ferait un petit generateur de contenu.

In [ ]:
selected_user = df_users_clean.iloc[0].to_dict()
selected_recommendations = engine.recommend(selected_user, n=3)

print("Profil selectionne")
print("Nom :", selected_user["name"])
print("Age :", selected_user["age"])
print("Interets :", selected_user["interests"])
print("Activites recentes :", selected_user["activity_log"][:4])

print("\nMessage genere :")
print("Bonjour", selected_user["name"] + ", voici des contenus adaptes a ton profil :")
for index, recommendation in enumerate(selected_recommendations, start=1):
    print(str(index) + ".", recommendation)

print("\nConclusion : le systeme utilise les interets, l'activite, la similarite entre utilisateurs et une logique sequentielle simple pour personnaliser les suggestions.")